Codigo para limpiar entorno, borrar archivos iniciales en bronze

In [0]:
# 00. Limpieza del esquema Bronze
print("Limpiando entorno...")

# Asegurar que el esquema existe
spark.sql("CREATE SCHEMA IF NOT EXISTS proyecto_iam_bronze")

# Lista de tablas a limpiar
tablas_a_borrar = [
    "identityx", 
    "planta_1", 
    "planta_2", 
    "matriz_modelamiento", 
    "gda_appgate_usuarios_grupos",
    "jira_usuarios_permisos" # Se deja preparada para cuando me des la estructura
]

for tabla in tablas_a_borrar:
    spark.sql(f"DROP TABLE IF EXISTS proyecto_iam_bronze.{tabla}")
    print(f"Tabla {tabla} eliminada (si existía).")

print("¡Entorno limpio y listo para generar nueva data!")

Limpiando entorno...
Tabla identityx eliminada (si existía).
Tabla planta_1 eliminada (si existía).
Tabla planta_2 eliminada (si existía).
Tabla matriz_modelamiento eliminada (si existía).
Tabla gda_appgate_usuarios_grupos eliminada (si existía).
Tabla jira_usuarios_permisos eliminada (si existía).
¡Entorno limpio y listo para generar nueva data!


In [0]:
# Reset de tablas para evitar duplicidad
tablas = [
    "proyecto_iam_bronze.identityx", 
    "proyecto_iam_bronze.planta_1", 
    "proyecto_iam_bronze.planta_2", 
    "proyecto_iam_bronze.matriz_modelamiento", 
    "proyecto_iam_bronze.gda_appgate_usuarios_grupos",
    "proyecto_iam_bronze.jira_usuarios_permisos"
]

for t in tablas:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

print("Tablas eliminadas. Por favor, vuelve a ejecutar las Celdas 1 a 5 una sola vez.")

Tablas eliminadas. Por favor, vuelve a ejecutar las Celdas 1 a 5 una sola vez.


Se comienza con la creacion de archivos con datos sinteticos pero que se asemejan a los archivos originales con los que se va a trabajar

In [0]:
%restart_python

In [0]:
# 01. Inicialización de Identidades Maestras (IdentityX) - Capa Bronze
%pip install faker
from faker import Faker
import pandas as pd
import random
import string
import re
import unicodedata

fake = Faker('es_CO')
num_usuarios = 2000 

# --- FUNCIONES DE LIMPIEZA ---
def normalizar_encabezado(nombre):
    n = nombre.lower()
    n = re.sub(r'[áéíóú]', lambda x: {'á':'a','é':'e','í':'i','ó':'o','ú':'u'}[x.group()], n)
    n = re.sub(r'[^a-z0-9]', '_', n)
    return re.sub(r'_+', '_', n).strip('_')

def quitar_tildes(texto):
    if not isinstance(texto, str): return texto
    # Reemplazar ñ por n explícitamente y quitar tildes/acentos
    texto = texto.replace('ñ', 'n').replace('Ñ', 'N')
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

# --- CONTROL DE USUARIOS ÚNICOS (SOLO LETRAS) ---
usuarios_generados = set()

def generar_usuario_unico(p_nombre, p_apellido):
    # Base inicial: 1ra letra del nombre + apellido (solo letras, minúsculas)
    base = quitar_tildes(p_nombre[:1] + p_apellido).lower()
    base = ''.join(filter(str.isalpha, base))
    
    usuario = base[:8] # Máximo 8 caracteres
    
    # Si no existe, lo usamos
    if usuario not in usuarios_generados:
        usuarios_generados.add(usuario)
        return usuario
    
    # Si YA EXISTE (Duplicado), usamos letras del alfabeto para diferenciarlo (no números)
    alfabeto = string.ascii_lowercase
    
    # Intento 1: Cambiar la última letra (ej. dgonzala, dgonzalb...)
    for letra in alfabeto:
        intento = (base[:7] + letra)
        if intento not in usuarios_generados:
            usuarios_generados.add(intento)
            return intento
            
    # Intento 2: Cambiar las dos últimas letras (ej. dgonzaaa, dgonzaab...)
    for l1 in alfabeto:
        for l2 in alfabeto:
            intento = (base[:6] + l1 + l2)
            if intento not in usuarios_generados:
                usuarios_generados.add(intento)
                return intento

# --- CONFIGURACIÓN DE NEGOCIO ---
proveedores = [("VEN001", "Konecta"), ("VEN002", "Teleperformance"), ("VEN003", "Axity"), ("VEN004", "Indra")]
roles_config = [(101, "Analista de Ciberseguridad", True), (102, "Operador de Monitoreo", True), 
                (103, "Especialista de Riesgos", True), (104, "Analista IAM", True), 
                (105, "Desarrollador Backend", False), (106, "Consultor Externo", True)]

encabezados_reales = ["Clase de Empleado (External Code)", "Clase de Empleado (Picklist Label)", "ID de empleado", "Nombre Completo", "Número de identificación", "Usuario", "Primer Apellido", "Segundo Apellido", "Tercer apellido /Apellido de casada", "Primer Nombre", "Segundo Nombre", "Tercer Nombre", "Departamento (ID Departamento)", "Departamento (Label)", "Rol de Negocio (ID Subclase Empleado)", "Rol de Negocio (Nombre)", "Position ID Posición", "Position Nombre Posición (Label)", "Jefe Inmediato", "Nombre del superior", "Compañia (ID Compañía)", "Compañia (Label)", "Empresa Proveedora/Temporal (Codigo empresa proveedora)", "Empresa Proveedora/Temporal (vendorName)", "Centro de Costos (ID Centro de Costos)", "Centro de Costos (Label)", "Subclase de empleado (Picklist Label)", "Región/Zona (Picklist Label)", "Acceso Lógico (Picklist Label)", "División (ID División)", "División (Label)", "Región/Zona (External Code)", "Subclase de empleado (External Code)", "ID de persona"]
columnas_tecnicas = [normalizar_encabezado(col) for col in encabezados_reales]

master_data = []
for i in range(num_usuarios):
    clase_code = random.choice([1, 4, 9, 9])
    clase_label = {1: "Interno", 4: "Aprendiz", 9: "Externo"}[clase_code]
    cedula = str(fake.random_number(digits=10, fix_len=True)) if random.random() > 0.05 else ''.join(random.choices(string.ascii_uppercase + string.digits, k=9))
    
    # Nombres limpios, sin tildes ni caracteres raros
    p_nombre = quitar_tildes(fake.first_name().replace(" ", ""))
    s_nombre = quitar_tildes(fake.first_name().replace(" ", ""))
    p_apellido = quitar_tildes(fake.last_name().replace(" ", ""))
    s_apellido = quitar_tildes(fake.last_name().replace(" ", ""))
    
    nombre_completo = f"{p_nombre} {s_nombre} {p_apellido} {s_apellido}".strip()
    nombre_superior = quitar_tildes(fake.name())
    
    # Generar el usuario de red con la nueva regla
    usuario_red = generar_usuario_unico(p_nombre, p_apellido)
    
    vend_cod, vend_name = (random.choice(proveedores) if clase_code == 9 else ("", "Directo"))
    rol_id, rol_name, _ = random.choice(roles_config)
    
    fila = [clase_code, clase_label, f"EMP{7000+i}", nombre_completo, cedula, usuario_red, p_apellido, s_apellido, "", p_nombre, s_nombre, "", f"DEP{random.randint(10,15)}", "Banca Digital", rol_id, rol_name, f"POS{i}", f"Posicion {rol_name}", f"JEF{random.randint(500,550)}", nombre_superior, "COMP_01", "Nequi/Bancolombia", vend_cod, vend_name, f"CC_{random.randint(2000,2100)}", "Centro de Costos", "Activo", "Antioquia", "Si", "DIV_01", "Banca Personas", "REG_COL", "SUB_EXT", f"IDP_{i}"]
    master_data.append(fila)

# Guardado en Delta Table
spark.createDataFrame(pd.DataFrame(master_data, columns=columnas_tecnicas)).write.format("delta").mode("overwrite").saveAsTable("proyecto_iam_bronze.identityx")

print(f"IdentityX generado con {num_usuarios} usuarios. Columna 'usuario' validada: solo letras, sin duplicados.")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
IdentityX generado con 2000 usuarios. Columna 'usuario' validada: solo letras, sin duplicados.


In [0]:
%sql
use catalog `workspace`; select * from `proyecto_iam_bronze`.`identityx` limit 100;

clase_de_empleado_external_code,clase_de_empleado_picklist_label,id_de_empleado,nombre_completo,numero_de_identificacion,usuario,primer_apellido,segundo_apellido,tercer_apellido_apellido_de_casada,primer_nombre,segundo_nombre,tercer_nombre,departamento_id_departamento,departamento_label,rol_de_negocio_id_subclase_empleado,rol_de_negocio_nombre,position_id_posicion,position_nombre_posicion_label,jefe_inmediato,nombre_del_superior,compa_ia_id_compa_ia,compa_ia_label,empresa_proveedora_temporal_codigo_empresa_proveedora,empresa_proveedora_temporal_vendorname,centro_de_costos_id_centro_de_costos,centro_de_costos_label,subclase_de_empleado_picklist_label,region_zona_picklist_label,acceso_logico_picklist_label,division_id_division,division_label,region_zona_external_code,subclase_de_empleado_external_code,id_de_persona
4,Aprendiz,EMP7000,Hugo Maria Valencia Martinez,1718302883,hvalenci,Valencia,Martinez,,Hugo,Maria,,DEP11,Banca Digital,103,Especialista de Riesgos,POS0,Posicion Especialista de Riesgos,JEF543,Johan Arnulfo Vargas,COMP_01,Nequi/Bancolombia,,Directo,CC_2009,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_0
1,Interno,EMP7001,Maria Miguel Serna Zambrano,5084838770,mserna,Serna,Zambrano,,Maria,Miguel,,DEP14,Banca Digital,103,Especialista de Riesgos,POS1,Posicion Especialista de Riesgos,JEF520,Alexander Perez,COMP_01,Nequi/Bancolombia,,Directo,CC_2050,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_1
1,Interno,EMP7002,Ingrid Martha Giraldo Orozco,2353225843,igiraldo,Giraldo,Orozco,,Ingrid,Martha,,DEP13,Banca Digital,102,Operador de Monitoreo,POS2,Posicion Operador de Monitoreo,JEF505,Valentina Ortiz,COMP_01,Nequi/Bancolombia,,Directo,CC_2039,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_2
1,Interno,EMP7003,Luz Cristian Jimenez Molano,7014004853,ljimenez,Jimenez,Molano,,Luz,Cristian,,DEP12,Banca Digital,103,Especialista de Riesgos,POS3,Posicion Especialista de Riesgos,JEF503,Edgar Oscar Avila Pardo,COMP_01,Nequi/Bancolombia,,Directo,CC_2096,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_3
9,Externo,EMP7004,Elisa Clara Martinez Mena,DNXBBCN8Z,emartine,Martinez,Mena,,Elisa,Clara,,DEP14,Banca Digital,104,Analista IAM,POS4,Posicion Analista IAM,JEF542,Fabian Ruiz Diaz,COMP_01,Nequi/Bancolombia,VEN002,Teleperformance,CC_2090,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_4
1,Interno,EMP7005,Ligia Laura Lopez Pabon,9808074633,llopez,Lopez,Pabon,,Ligia,Laura,,DEP12,Banca Digital,103,Especialista de Riesgos,POS5,Posicion Especialista de Riesgos,JEF509,Luz Olga Diaz,COMP_01,Nequi/Bancolombia,,Directo,CC_2041,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_5
4,Aprendiz,EMP7006,Daniel Aura Guevara Franco,3969945275,dguevara,Guevara,Franco,,Daniel,Aura,,DEP15,Banca Digital,106,Consultor Externo,POS6,Posicion Consultor Externo,JEF516,Jose Acevedo Arevalo,COMP_01,Nequi/Bancolombia,,Directo,CC_2010,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_6
1,Interno,EMP7007,Humberto Jose Serrano Lopez,8165713200,hserrano,Serrano,Lopez,,Humberto,Jose,,DEP11,Banca Digital,105,Desarrollador Backend,POS7,Posicion Desarrollador Backend,JEF530,Gabriel Mauricio Sanchez,COMP_01,Nequi/Bancolombia,,Directo,CC_2043,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_7
4,Aprendiz,EMP7008,Maria Lucila Rojas Avila,8679958905,mrojas,Rojas,Avila,,Maria,Lucila,,DEP15,Banca Digital,101,Analista de Ciberseguridad,POS8,Posicion Analista de Ciberseguridad,JEF542,Elcy Martinez,COMP_01,Nequi/Bancolombia,,Directo,CC_2093,Centro de Costos,Activo,Antioquia,Si,DIV_01,Banca Personas,REG_COL,SUB_EXT,IDP_8
4,Aprendiz,EMP7009,Daniela John Beltran Sandoval,4344657369,dbeltran,Beltran,Sandoval,,Daniela,John,,DEP15,Banca Digital,106,Consultor Externo,POS9,Posicion Consultor Externo,JEF513,Karen Delgado Diaz,COMP_01,Nequi/Bancolombia,,Dire

generar los archivos plantas

In [0]:
# 02. Generación de Puentes de Identidad (Plantas BPO 1 y 2) - Capa Bronze
df_master = spark.table("proyecto_iam_bronze.identityx").toPandas()

# Planta 1: Konecta
df_konecta = df_master[df_master['empresa_proveedora_temporal_vendorname'] == 'Konecta']
p1_data = [[row['numero_de_identificacion'], row['nombre_completo'], f"{row['usuario']}@konecta.com", row['usuario']] for _, row in df_konecta.iterrows()]
spark.createDataFrame(pd.DataFrame(p1_data, columns=["cedula", "nombre", "correo", "usuario_banco"])).write.format("delta").mode("overwrite").saveAsTable("proyecto_iam_bronze.planta_1")

# Planta 2: Teleperformance
df_tp = df_master[df_master['empresa_proveedora_temporal_vendorname'] == 'Teleperformance']
p2_data = [[row['nombre_completo'], row['numero_de_identificacion'], row['usuario'], f"{row['usuario']}@teleperformance.com"] for _, row in df_tp.iterrows()]
spark.createDataFrame(pd.DataFrame(p2_data, columns=["nombre", "cc", "usuario_banco", "correo_coorporativo"])).write.format("delta").mode("overwrite").saveAsTable("proyecto_iam_bronze.planta_2")

print("Plantas BPO generadas.")

Plantas BPO generadas.


matriz de modelamiento appgate jira

In [0]:
# 03. Construcción de la Matriz de Modelamiento Maestra (Governance) - Capa Bronze
roles_uo = spark.table("proyecto_iam_bronze.identityx").select("departamento_id_departamento", "departamento_label", "rol_de_negocio_id_subclase_empleado", "rol_de_negocio_nombre").distinct().toPandas()
matriz_data = []

# Aplicaciones reducidas a Appgate y Jira
aplicaciones = ["Appgate", "Jira"]

for _, row in roles_uo.iterrows():
    for app in aplicaciones:
        rol_tec = ""
        # Lógica definida: Appgate igual al negocio, Jira con prefijos
        if app == "Appgate":
            rol_tec = row['rol_de_negocio_nombre']
        elif app == "Jira":
            rol_tec = f"Jira_{row['rol_de_negocio_nombre'].replace(' ', '_')}"
            
        matriz_data.append([app, row['departamento_id_departamento'], row['departamento_label'], row['rol_de_negocio_id_subclase_empleado'], row['rol_de_negocio_nombre'], rol_tec])

spark.createDataFrame(pd.DataFrame(matriz_data, columns=["aplicacion", "id_uo", "unidad_organizativa", "id_rol_negocio", "rol_de_negocio", "roles"])).write.format("delta").mode("overwrite").saveAsTable("proyecto_iam_bronze.matriz_modelamiento")
print("Matriz de Modelamiento creada (Appgate y Jira).")

Matriz de Modelamiento creada (Appgate y Jira).


creacion de archivo extraccion appgate

In [0]:
# 04. Generación de Extracto Appgate con Inyección de Anomalías - Capa Bronze
appgate_rows = []
for _, user in df_master.sample(frac=0.7).iterrows(): # 70% del personal tiene Appgate
    
    # Decidimos si este usuario tendrá los permisos correctos (90%) o si le inyectamos una anomalía (10%)
    if random.random() > 0.10:
        rol_final = user['rol_de_negocio_nombre'] # Correcto: Igual al rol de negocio
    else:
        rol_final = "Permiso_Incorrecto_No_Modelado" # Incorrecto: Salta la alarma en el cruce

    # Simulando múltiples entitlements (líneas duplicadas en CSV)
    for ent in ["VPN_Access_Prod", "Network_Drive_Read"]:
        appgate_rows.append([user['usuario'], rol_final, ent])

spark.createDataFrame(pd.DataFrame(appgate_rows, columns=["usuario", "politica_grupo_appgate", "entitlement"])).write.format("delta").mode("overwrite").saveAsTable("proyecto_iam_bronze.gda_appgate_usuarios_grupos")

print("Extracto de Appgate generado.")

Extracto de Appgate generado.


creacion del archivo extraccion jira

In [0]:
# 05. Generación de Extracto Jira (Basado en Correo) - Capa Bronze
import pandas as pd
import random

# 1. Cargar datos maestros para sincronizar
df_master = spark.table("proyecto_iam_bronze.identityx").toPandas()
df_matriz_jira = spark.table("proyecto_iam_bronze.matriz_modelamiento").filter("aplicacion = 'Jira'").toPandas()

jira_rows = []

# Tomamos una muestra del 75% de los usuarios para que tengan cuenta en Jira
for _, user in df_master.sample(frac=0.75).iterrows():
    user_red = user['usuario']
    empresa = user['empresa_proveedora_temporal_vendorname']
    rol_negocio = user['rol_de_negocio_nombre']
    
    # --- LÓGICA DE CORREO SINCRONIZADO CON LAS PLANTAS ---
    if empresa == 'Konecta':
        correo = f"{user_red}@konecta.com"
    elif empresa == 'Teleperformance':
        correo = f"{user_red}@teleperformance.com"
    else:
        # Para internos y otros proveedores
        correo = f"{user_red}@nequi.com.co"
    
    # --- LÓGICA DE ROL TÉCNICO (CON ANOMALÍAS) ---
    # Buscamos el rol correcto en la matriz
    row_matriz = df_matriz_jira[df_matriz_jira['rol_de_negocio'] == rol_negocio]
    
    if not row_matriz.empty and random.random() > 0.10:
        # Escenario Correcto (90%): Asignamos el rol que dice la matriz
        rol_jira = row_matriz.iloc[0]['roles']
    else:
        # Escenario Anomalía (10%): Asignamos un rol de administrador o uno aleatorio
        roles_posibles = ["Jira_Administrators", "Jira_System_Config", "Jira_Power_User"]
        rol_jira = random.choice(roles_posibles)

    jira_rows.append([correo, rol_jira])

# 2. Crear DataFrame con los encabezados solicitados
columnas_jira = ["correo_electronico", "rol_tecnico_en_jira"]
df_jira_final = pd.DataFrame(jira_rows, columns=columnas_jira)

# 3. Guardar en la capa Bronze
spark.createDataFrame(df_jira_final).write.format("delta").mode("overwrite").saveAsTable("proyecto_iam_bronze.jira_usuarios_permisos")

print("---")
print("¡EXTRACTO DE JIRA GENERADO EXITOSAMENTE!")
print(f"Registros creados: {len(jira_rows)}")
print("Sincronización: Los correos coinciden con las Plantas BPO.")
print("Anomalías: 10% de los registros tienen roles fuera de la matriz.")
print("---")

---
¡EXTRACTO DE JIRA GENERADO EXITOSAMENTE!
Registros creados: 1500
Sincronización: Los correos coinciden con las Plantas BPO.
Anomalías: 10% de los registros tienen roles fuera de la matriz.
---


In [0]:
# Bloque para auditar volúmenes de la Capa Bronze
print("--- Auditoría de Volúmenes (Capa Bronze) ---")

tablas = [
    ("identityx", "Usuarios Únicos (Personas)"),
    ("gda_appgate_usuarios_grupos", "Accesos en Appgate (Permisos)"),
    ("jira_usuarios_permisos", "Accesos en Jira (Permisos)"),
    ("matriz_modelamiento", "Reglas de la Matriz (Configuración)")
]

for tabla, descripcion in tablas:
    count = spark.table(f"proyecto_iam_bronze.{tabla}").count()
    print(f"{descripcion} ({tabla}): {count} registros")

--- Auditoría de Volúmenes (Capa Bronze) ---
Usuarios Únicos (Personas) (identityx): 2000 registros
Accesos en Appgate (Permisos) (gda_appgate_usuarios_grupos): 2800 registros
Accesos en Jira (Permisos) (jira_usuarios_permisos): 1500 registros
Reglas de la Matriz (Configuración) (matriz_modelamiento): 72 registros


INICIO CAPA SILVER

In [0]:
# CAPA SILVER: Maestro de Accesos Unificado
# Objetivo: Consolidar Jira y Appgate, enriquecer con datos de IdentityX

from pyspark.sql import functions as F

# 1. Crear el esquema Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS proyecto_iam_silver")

# 2. Procesar APPGATE (ya tiene usuario directo)
df_appgate = spark.table("proyecto_iam_bronze.gda_appgate_usuarios_grupos") \
    .select(
        F.lit("Appgate").alias("aplicacion"),
        F.col("usuario").alias("usuario_red"),
        F.col("politica_grupo_appgate").alias("rol_tecnico_asignado")
    ).distinct()

# 3. Procesar JIRA (necesita traducción de correo a usuario_red)
# 3.1. Cargar Jira y plantas
df_jira = spark.table("proyecto_iam_bronze.jira_usuarios_permisos")
df_planta1 = spark.table("proyecto_iam_bronze.planta_1")  # Konecta
df_planta2 = spark.table("proyecto_iam_bronze.planta_2")  # Teleperformance

# 3.2. Unificar plantas (normalizar estructura)
df_planta1_norm = df_planta1.select(
    F.col("correo").alias("correo"),
    F.col("usuario_banco").alias("usuario_red")
)

df_planta2_norm = df_planta2.select(
    F.col("correo_coorporativo").alias("correo"),
    F.col("usuario_banco").alias("usuario_red")
)

# 3.3. Unir plantas
df_plantas_unidas = df_planta1_norm.unionAll(df_planta2_norm)

# 3.4. Traducir Jira: correo -> usuario_red
df_jira_traducido = df_jira.join(
    df_plantas_unidas,
    df_jira.correo_electronico == df_plantas_unidas.correo,
    "left"
).select(
    F.lit("Jira").alias("aplicacion"),
    F.coalesce(
        F.col("usuario_red"),
        F.regexp_extract(F.col("correo_electronico"), r'^([^@]+)', 1)
    ).alias("usuario_red"),
    F.col("rol_tecnico_en_jira").alias("rol_tecnico_asignado")
).distinct()

# 4. UNIFICAR Appgate + Jira
df_accesos_unificados = df_appgate.unionAll(df_jira_traducido)

# 5. ENRIQUECER con datos de IdentityX (INCLUIR departamento_id para el join correcto)
df_identityx = spark.table("proyecto_iam_bronze.identityx").select(
    F.col("usuario").alias("usuario_red"),
    F.col("nombre_completo"),
    F.col("numero_de_identificacion").alias("cedula"),
    F.col("rol_de_negocio_nombre").alias("rol_de_negocio"),
    F.col("departamento_id_departamento").alias("departamento_id"),  # ✅ AGREGADO
    F.col("departamento_label").alias("departamento"),
    F.col("clase_de_empleado_picklist_label").alias("tipo_empleado")
)

df_maestro_accesos = df_accesos_unificados.join(
    df_identityx,
    "usuario_red",
    "left"  # Left join para detectar huérfanos
).select(
    "aplicacion",
    "usuario_red",
    "nombre_completo",
    "cedula",
    "tipo_empleado",
    "departamento_id",  # ✅ AGREGADO
    "departamento",
    "rol_de_negocio",
    "rol_tecnico_asignado"
)

# 6. Guardar en Silver con overwriteSchema para permitir cambio de esquema
df_maestro_accesos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("proyecto_iam_silver.maestro_accesos")

print("✅ Capa Silver creada exitosamente (con departamento_id).")
print(f"Total de accesos unificados: {df_maestro_accesos.count()}")
print("\nDesglose por aplicación:")
df_maestro_accesos.groupBy("aplicacion").count().show()

✅ Capa Silver creada exitosamente (con departamento_id).
Total de accesos unificados: 2900

Desglose por aplicación:
+----------+-----+
|aplicacion|count|
+----------+-----+
|   Appgate| 1400|
|      Jira| 1500|
+----------+-----+



In [0]:
# CAPA GOLD: Reporte de Auditoría con Clasificación de Estados (FIX FINAL)
# Objetivo: Comparar accesos reales vs matriz de modelamiento y detectar anomalías
# FIX: Usar departamento_id (ID) en lugar del label para evitar producto cartesiano

from pyspark.sql import functions as F

# 1. Crear el esquema Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS proyecto_iam_gold")

# 2. Cargar datos de Silver y Matriz con alias
df_accesos = spark.table("proyecto_iam_silver.maestro_accesos").alias("acc")
df_matriz = spark.table("proyecto_iam_bronze.matriz_modelamiento").alias("mat")

# 3. CRUZAR CON DEPARTAMENTO_ID (no el label) para match 1:1
df_auditoria = df_accesos.join(
    df_matriz,
    (F.col("acc.aplicacion") == F.col("mat.aplicacion")) & 
    (F.col("acc.rol_de_negocio") == F.col("mat.rol_de_negocio")) &
    (F.col("acc.departamento_id") == F.col("mat.id_uo")),  # ✅ USAR ID, NO LABEL
    "left"  # Left join para detectar roles no modelados
)

# 4. Aplicar lógica de clasificación
df_reporte = df_auditoria.withColumn(
    "estado_auditoria",
    F.when(
        F.col("acc.nombre_completo").isNull(), 
        F.lit("Cuenta Huérfana")
    ).when(
        F.col("mat.roles").isNull(), 
        F.lit("No Modelado")
    ).when(
        F.col("acc.rol_tecnico_asignado") == F.col("mat.roles"), 
        F.lit("Modelado Correcto")
    ).otherwise(
        F.lit("Anomalía (Acceso Incorrecto)")
    )
).select(
    F.col("acc.aplicacion").alias("aplicacion"),
    F.col("acc.usuario_red").alias("usuario_red"),
    F.col("acc.nombre_completo").alias("nombre_completo"),
    F.col("acc.cedula").alias("cedula"),
    F.col("acc.tipo_empleado").alias("tipo_empleado"),
    F.col("acc.departamento_id").alias("departamento_id"),
    F.col("acc.departamento").alias("departamento"),
    F.col("acc.rol_de_negocio").alias("rol_de_negocio"),
    F.col("acc.rol_tecnico_asignado").alias("rol_tecnico_asignado"),
    F.col("mat.roles").alias("rol_esperado_segun_matriz"),
    "estado_auditoria",
    F.current_timestamp().alias("fecha_auditoria")
)

# 5. Guardar en Gold con overwriteSchema
df_reporte.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("proyecto_iam_gold.reporte_auditoria")

print("✅ Capa Gold creada exitosamente (FIX FINAL - Join por ID).")
print(f"\nTotal de accesos auditados: {df_reporte.count()} (esperado: ~2900)")
print("\n" + "="*70)
print("RESUMEN DE AUDITORÍA IAM")
print("="*70)

# Resumen por estado
df_resumen = df_reporte.groupBy("aplicacion", "estado_auditoria").count().orderBy("aplicacion", F.desc("count"))
df_resumen.show(truncate=False)

# Totales generales
print("\nTOTALES POR ESTADO (Consolidado):")
df_reporte.groupBy("estado_auditoria").count().orderBy(F.desc("count")).show(truncate=False)

# Identificar anomalías críticas
anomalias = df_reporte.filter(F.col("estado_auditoria").isin(["Anomalía (Acceso Incorrecto)", "Cuenta Huérfana"])).count()
print(f"\n⚠️  ALERTAS CRÍTICAS DETECTADAS: {anomalias} registros")

if anomalias > 0:
    print("\n🔴 Se encontraron anomalías que requieren revisión inmediata.")
    print("   - Cuentas Huérfanas: Accesos sin identidad en el maestro")
    print("   - Anomalías de Acceso: Permisos que no coinciden con la matriz de gobierno")
else:
    print("\n🟢 No se detectaron anomalías críticas. El entorno está alineado con la matriz de gobierno.")

✅ Capa Gold creada exitosamente (FIX FINAL - Join por ID).

Total de accesos auditados: 2900 (esperado: ~2900)

RESUMEN DE AUDITORÍA IAM
+----------+----------------------------+-----+
|aplicacion|estado_auditoria            |count|
+----------+----------------------------+-----+
|Appgate   |Modelado Correcto           |1257 |
|Appgate   |Anomalía (Acceso Incorrecto)|143  |
|Jira      |Modelado Correcto           |1364 |
|Jira      |Anomalía (Acceso Incorrecto)|136  |
+----------+----------------------------+-----+


TOTALES POR ESTADO (Consolidado):
+----------------------------+-----+
|estado_auditoria            |count|
+----------------------------+-----+
|Modelado Correcto           |2621 |
|Anomalía (Acceso Incorrecto)|279  |
+----------------------------+-----+


⚠️  ALERTAS CRÍTICAS DETECTADAS: 279 registros

🔴 Se encontraron anomalías que requieren revisión inmediata.
   - Cuentas Huérfanas: Accesos sin identidad en el maestro
   - Anomalías de Acceso: Permisos que no coincide

In [0]:
%sql
SELECT aplicacion, estado_auditoria, COUNT(*) as total 
FROM proyecto_iam_gold.reporte_auditoria 
GROUP BY 1, 2 ORDER BY 1, 3 DESC

aplicacion,estado_auditoria,total
Appgate,Modelado Correcto,7542
Appgate,Anomalía (Acceso Incorrecto),858
Jira,Modelado Correcto,8184
Jira,Anomalía (Acceso Incorrecto),816
